In [ ]:
import os

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

import numpy as np
import pandas as pd
import copy
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import statsmodels.api as sm
from skrub import GapEncoder
from sklearn.model_selection import train_test_split
import shap
from transformers import AutoTokenizer
from statsmodels.stats.multitest import multipletests
import plotnine as p9
from matplotlib import pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from collections import defaultdict

from src.clustering.settings import best_pipeline, final_cluster_name, clinical_endpoints, rename_cols_dict, RANDOM_STATE, timepoints

plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
endpoints = [endpoint["description"] for endpoint in clinical_endpoints.values()]

In [ ]:
embedding_clusters = os.path.join("results_v2_permodality", "clustering", "embedding_clusters.csv")
y_clusters = pd.read_csv(embedding_clusters, index_col=0)[best_pipeline["name"]]
embedding_clusters = y_clusters.replace({0: "Low-risk", 1: "High-risk"}).astype(object).squeeze().rename(final_cluster_name)
print(f"{embedding_clusters.shape}")
embedding_clusters.head()

In [ ]:
X = []
text_types = ["consultation_notes", "pathology_reports"]
text_folder = os.path.join("..", "HNC_Reports")
for patient_id in embedding_clusters.index:
    text_dict = {
        "patient_id": patient_id,
    }
    for text_type in text_types:
        text_path = os.path.join(text_folder, text_type, f"{patient_id}.txt")
        if os.path.exists(text_path):
            with open(text_path, "r") as f:
                text = f.read().replace('\n', ' ')
        else:
            text = np.nan
        text_dict[text_type] = text
    X.append(text_dict)
X = pd.DataFrame(X).set_index("patient_id")
X = X.rename(columns={"consultation_notes": "cons", "pathology_reports": "path"})
X = X.loc[embedding_clusters.index]
text_types = X.columns.to_list()
print(f"{X.shape}")
X.head()

In [ ]:
pipeline = make_pipeline(
    GapEncoder(n_components=30, random_state=42).set_output(transform="pandas"),
    StandardScaler().set_output(transform="pandas"),
    LogisticRegression(l1_ratio=1., random_state=42, solver="saga", max_iter=10000),
)

In [ ]:
for text_type in text_types:
    X[text_type] = X[text_type] \
        .str.replace("theunknown", "the unknown") \
        .str.replace("protocolnumber", "protocol number") \
        .str.replace("aappointments", "appointments") \
        .str.replace("alalveolar", "alveolar") \
        .str.replace("______", " ")

In [ ]:
th_pval = - np.log10(0.05)
Xs_transformed = []
Xs = []
cv_res = {}
encs = []
scores_list = []
for text_type in text_types:
    print(text_type)
    X_text = X[text_type]
    y_text = y_clusters[X_text.notnull()]
    X_text = X_text[X_text.notnull()]
    pd.testing.assert_index_equal(X_text.index, y_text.index)
    scores = cross_val_score(
        estimator=pipeline,
        X=X_text,
        y=y_text,
        n_jobs=5,
        cv=5,
        scoring="matthews_corrcoef",
    )
    print(scores.round(2))
    scores_list.append([{"text_type": text_type, "i": i, "score": score} for i, score in enumerate(scores)])
    to_print = f"{scores.mean().round(2)} \u00B1 {scores.std().round(2)}"
    cv_res[text_type] = to_print
    print(to_print)

    pipeline.fit(X_text, y_text)
    X_text_with_constant = pipeline[:-1].transform(X_text)
    print(f"{X_text_with_constant.shape}")
    selected_indices = pipeline[-1].coef_.nonzero()[1]
    X_text_with_constant = X_text_with_constant.iloc[:, selected_indices]
    print(f"{X_text_with_constant.shape}")
    logit_model = sm.Logit(y_text, sm.add_constant(X_text_with_constant))
    result = logit_model.fit(method='bfgs', maxiter=1000)
    encs.append(copy.deepcopy(pipeline[0]))
    Xs_transformed.append(X_text_with_constant)

    y_text = y_text.replace({0: "Low-risk", 1: "High-risk"})
    word_coefs = pd.DataFrame([
        result.params.index,
        result.params.to_list(),
        result.pvalues],
        index=["Word", "Coefficient", "pvalue"]).T
    word_coefs = word_coefs[word_coefs["Word"].isin(X_text_with_constant.columns)]
    word_coefs["Component"] = list(range(len(word_coefs)))
    word_coefs["Word"] = word_coefs["Word"].str.removeprefix(f"{text_type}: ").str.split(", ").apply(lambda x: x[0])
    word_coefs = word_coefs.assign(text_type=text_type)
    Xs.append(word_coefs)
    print("-"*80)
Xs = pd.concat(Xs).reset_index(drop=True)
Xs["Coefficient"] = Xs["Coefficient"].astype(float)
Xs["pvalue"] = Xs["pvalue"].astype(float)
Xs["score_text"] = Xs["text_type"].apply(lambda x: f"MCC = {cv_res[x]}")
Xs["Abs Coefficient"] = Xs["Coefficient"].abs()
Xs["-log(padj)"] = - np.log10(multipletests(Xs["pvalue"], method="fdr_bh")[1])
Xs["adjp"] = np.where(Xs["-log(padj)"].gt(th_pval), f"adjp<0.05", "adjp>=0.05")
Xs = Xs.sort_values("Abs Coefficient", ascending=False)

In [ ]:
# x,y = "Ratio_log", "-log(padj)"
# facet_cols = "text_type"
# fill_by = "count"

# p_vol = (
#     p9.ggplot(Xs, p9.aes(x=x, y=y))
#     + p9.geom_text(p9.aes(label="Word", color="adjp"))
#     + p9.geom_hline(yintercept=th_pval, linetype="dashed")
#     + p9.geom_label(p9.aes(label="score_text"), data=Xs.drop_duplicates(subset="score_text"), x=-1, y=9, inherit_aes=False)
#     + p9.facet_wrap(facet_cols, scales="free_x")
#     + p9.scale_x_continuous(expand=(0.15, 0, 0.6, 0))
#     + p9.theme_classic()
#     + p9.theme(strip_background=p9.element_blank(), legend_position="top")
#     + p9.labs(x="log(High/Low risk)", tag="a", title="Volcano plot for words")
# )
# p_vol

In [ ]:
data_to_show = pd.concat([Xs[Xs["text_type"].eq(text_type)].sort_values("Abs Coefficient", ascending=False).iloc[:10] for text_type in text_types])
data_to_show["Component"] = pd.Categorical(data_to_show["Component"], data_to_show["Component"].unique()[::-1], ordered=True)
for text_type in text_types:
    mask = data_to_show["text_type"].eq(text_type)
    data_to_show.loc[mask, "posx"] = data_to_show[mask]["Coefficient"].min() * .7
p_coeff = (
    p9.ggplot(data_to_show, p9.aes(x="Component", y="Coefficient", fill="adjp"))
    + p9.geom_col(width=0.85)
    + p9.geom_text(p9.aes(y="posx", label="score_text"), data=data_to_show.drop_duplicates(subset="score_text"), x=7, inherit_aes=False)
    + p9.geom_label(p9.aes(label="Word"), y=-0.03, ha="left", fill="white")
    + p9.facet_wrap("text_type", scales="free", nrow=1)
    + p9.coord_flip()
    + p9.scale_fill_brewer(type="qualitative", palette="Pastel1", direction=1)
    + p9.theme_classic()
    + p9.theme(legend_position=(0.05, 0.1), legend_title=p9.element_blank(), strip_background=p9.element_blank(), legend_box_spacing=0.001,
               plot_margin_bottom=0, axis_title_x=p9.element_text(margin={'t': -0.5, 'units': 'pt'}))
    + p9.labs(tag="a", title="Cluster classification using unimodal text")
)
p_coeff

In [ ]:
source_data = pd.DataFrame([score for scores in scores_list for score in scores])
with pd.ExcelWriter(os.path.join("source_data", "Source Data S5.xlsx"), engine='openpyxl') as writer:
    data_to_show[["Word", "Coefficient", "pvalue", "Component", "text_type", "-log(padj)"]].to_excel(writer, sheet_name='Figure 6a', index=False)
    source_data.to_excel(writer, sheet_name='Figure 6a (metrics)', index=False)

In [ ]:
component_list = []
top_words = 5
for enc,text_type in zip(encs, text_types):
    vectorizer = CountVectorizer()
    vectorizer.fit(list(enc.H_dict_.keys()))
    vocabulary = np.array(vectorizer.get_feature_names_out())
    encoding = enc._transform(vocabulary, np.zeros(len(vocabulary), dtype="bool"))
    encoding = abs(encoding)
    word_topic_weights = encoding / np.sum(encoding, axis=1, keepdims=True)
    n_components = word_topic_weights.shape[1]
    for i in range(n_components):
        component_weights = word_topic_weights[:, i]
        top_indices = np.argsort(-component_weights)[:top_words]
        for idx in top_indices:
            component_dict = {
                "text_type": text_type,
                "idx": i,
                "Word": vocabulary[idx],
                "Weight": component_weights[idx],
            }
            component_list.append(component_dict)
component_df = pd.DataFrame(component_list)
component_df["Component"] = "Component " + component_df["idx"].astype(str)
component_df["Word_idx"] = component_df["Word"] + "___" + component_df["idx"].astype(str)
top_c = pd.concat([Xs[Xs["text_type"].eq(text_type)].iloc[:2] for text_type in text_types])
component_df = component_df[
    (component_df["text_type"].eq(text_types[0]) & component_df["idx"].isin(top_c[top_c["text_type"].eq(text_types[0])]["Component"])) |
    (component_df["text_type"].eq(text_types[1]) & component_df["idx"].isin(top_c[top_c["text_type"].eq(text_types[1])]["Component"]))
]
component_df["Component"] = pd.Categorical(component_df["Component"], component_df["Component"].unique(), ordered=True)

x,y = "Word", "Weight"
facet_cols = ["text_type", "Component"]
p_comp = (
    p9.ggplot(component_df, p9.aes(x="reorder(Word_idx, Weight)", y=y))
    + p9.geom_col(fill="lightblue")
    + p9.facet_wrap(facet_cols, scales="free_y", nrow=1, labeller=p9.labeller(multi_line=False))
    + p9.scale_x_discrete(labels=lambda labels: [str(l).split("___")[0] for l in labels])
    + p9.coord_flip()
    + p9.theme_classic()
    + p9.theme(strip_background=p9.element_blank(), axis_title_y=p9.element_blank(), plot_margin_bottom=0, axis_title_x=p9.element_text(margin={'t': -0.5, 'units': 'pt'}))
    + p9.labs(tag="b", title="Word importance per component")
)
p_comp

In [ ]:
with pd.ExcelWriter(os.path.join("source_data", "Source Data S5.xlsx"), engine='openpyxl', mode="a") as writer:
    component_df[["text_type", "Word", "Weight", "Component"]].to_excel(writer, sheet_name='Figure 6b', index=False)

In [ ]:
# import seaborn as sns
# import matplotlib.pyplot as plt
# matrix = X_transformed.corr()
# matrix = matrix.reset_index(drop=True)
# matrix.columns = range(matrix.shape[1])
# plt.figure(figsize=(18, 8))
# sns.heatmap(
#     matrix,
#     annot=True,  # Shows the correlation coefficients in each cell
#     cmap="coolwarm",  # Diverging colormap (blue = negative, red = positive)
#     vmin=-1,  # Set minimum value for the color scale
#     vmax=1,  # Set maximum value for the color scale
#     fmt=".2f",  # Format values to 2 decimal places
# ).figure

In [ ]:
# import scipy.stats as stats
# from sklearn.metrics import roc_auc_score

# freq_df = []
# for text_type, X_transformed in zip(text_types, Xs_transformed):
#     for component_id,(colname, component) in enumerate(X_transformed.items()):
#         for label in embedding_clusters.unique():
#             mask = embedding_clusters == label
#             component_mean = component[mask].mean()
#             freq_dict = {
#                 "text_type": text_type,
#                 "Words": colname,
#                 "idx": component_id,
#                 "label": label,
#                 "mean": component_mean,
#                 "pval": stats.ttest_ind(component[mask], component[~mask])[1],
#                 "power": roc_auc_score(embedding_clusters[component.index], component),
#             }
#             freq_df.append(freq_dict)
# freq_df = pd.DataFrame(freq_df)
# freq_df = freq_df[
#     (freq_df["text_type"].eq("notes") & freq_df["idx"].eq(5)) |
#     (freq_df["text_type"].eq("notes") & freq_df["idx"].eq(28)) |
#     (freq_df["text_type"].eq("path") & freq_df["idx"].eq(25)) |
#     (freq_df["text_type"].eq("path") & freq_df["idx"].eq(17))
# ]
# freq_df

In [ ]:
# freq_df = []
# for text_type in text_types:
#     X_text = X[text_type]
#     component_text = component_df[component_df["text_type"].eq(text_type)]
#     for component in component_text["Component"].unique():
#         words = component_text[component_text["Component"].eq(component)]["Word"].to_list()
#         words = "|".join(words)
#         for label in embedding_clusters.unique():
#             mask = embedding_clusters == label
#             count = X_text[mask].dropna().str.count(words).mean()
#             freq_dict = {
#                 "text_type": text_type,
#                 "Component": component,
#                 "label": label,
#                 "Freq": count,
#             }
#             freq_df.append(freq_dict)
# freq_df = pd.DataFrame(freq_df)
# freq_df["Freq"] = freq_df["Freq"].mul(100).round(1)
# freq_df

In [ ]:
clinical_data = os.path.join("outcomes", "all_fields_processed.csv")
clinical_data = pd.read_csv(clinical_data)
clinical_data = clinical_data.rename(columns=rename_cols_dict)
clinical_data_with_clusters = clinical_data.merge(y_clusters.reset_index(name=final_cluster_name).rename(columns={'index': 'patient_id'}), on="patient_id")
print(f"{clinical_data.shape=}")
clinical_data.head()

In [ ]:
import shapiq
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
import sklearn
sklearn.set_config(transform_output="pandas")


shap_list = []
for clinical_endpoint_idx, clinical_endpoint in enumerate(clinical_endpoints):
    t = clinical_endpoints[clinical_endpoint]["t"]
    e = clinical_endpoints[clinical_endpoint]["e"]
    description = clinical_endpoints[clinical_endpoint]["description"]
        
    clinical_data_endpoint = clinical_data_with_clusters[
        clinical_data_with_clusters[t].notna() &
        clinical_data_with_clusters[e].notna()
        ]

    if description == "Overall Survival":
        X_vars = [final_cluster_name, "Age"]
        transformer = ColumnTransformer(
            transformers=[
                ('num', make_pipeline(
                    SimpleImputer(strategy="mean", add_indicator=True),
                    ), ["Age"]),
                ],
            remainder='passthrough'
        )
    elif description == "Distant Failure":
        X_vars = [final_cluster_name, "N-stage", "T-stage", "HPV Status"]
        clinical_data_endpoint["T-stage"] = clinical_data_endpoint["T-stage"].str[1].astype(float)
        clinical_data_endpoint["N-stage"] = clinical_data_endpoint["N-stage"].str[1].astype(float)
        clinical_data_endpoint["HPV Status"] = clinical_data_endpoint["HPV Status"].replace("Negative", 0).replace("Positive", 1)
        transformer = ColumnTransformer(
            transformers=[
                ('cat', make_pipeline(
                    SimpleImputer(strategy="most_frequent", add_indicator=True),
                    ),
                    ["HPV Status"]
                    ),
                ('num', make_pipeline(
                    SimpleImputer(strategy="mean", add_indicator=True),
                    ), ["T-stage", "N-stage"]),
                ],
            remainder='passthrough'
        )
    else:
        continue

    clinical_data_endpoint = clinical_data_endpoint.sort_values(by=t, ascending=False)
    clinical_data_endpoint = clinical_data_endpoint.drop_duplicates(subset='patient_id', keep='first')
    clinical_data_endpoint = clinical_data_endpoint.set_index("patient_id")
    X = clinical_data_endpoint[X_vars]


    for timepoint in [60, 120]:
        conditions = [
            (clinical_data_endpoint[t] <= timepoint) & (clinical_data_endpoint[e] == 1),
            (clinical_data_endpoint[t] > timepoint),
            (clinical_data_endpoint[t] <= timepoint) & (clinical_data_endpoint[e] == 0),
        ]
        choices = [1, 0, np.nan]
    
        binary_outcome = np.select(conditions, choices)
        binary_outcome = pd.Series(binary_outcome, index=clinical_data_endpoint.index)
        mask = binary_outcome.isna()
        binary_outcome = binary_outcome.loc[~mask]
        X_endpoint = X.loc[~mask]

        X_train, X_test, y_train, y_test = train_test_split(X_endpoint, binary_outcome, test_size=0.2, random_state=RANDOM_STATE, stratify=binary_outcome, shuffle=True)

        model = make_pipeline(
            transformer,
            RandomForestClassifier(random_state=RANDOM_STATE),
        )
        model.fit(X_train, y_train)
        model_predict = lambda x: model.predict(pd.DataFrame(x, columns=X_train.columns))

        explainer = shapiq.TabularExplainer(model_predict, data=X_test.to_numpy(), index="SII", max_order=2, random_state=RANDOM_STATE)
        explanations = explainer.explain_X(X_test.to_numpy(), budget=200)

        aggregated_scores = defaultdict(list)
        for exp in explanations:
            for interaction_tuple, val in exp.dict_values.items():
                aggregated_scores[interaction_tuple].append(abs(val))
        parsed_data = []
        for interaction_tuple, abs_vals_list in aggregated_scores.items():
            if not interaction_tuple:
                continue
            names = [X_vars[i] for i in interaction_tuple]
            label_str = " × ".join(names)
            order = f"{len(names)}-order"
            parsed_data.append({
                "interaction": label_str,
                "mean_abs_value": np.mean(abs_vals_list),
                "order": order
            })
        df_plot = pd.DataFrame(parsed_data)
        df_plot["endpoint"] = description
        df_plot["timepoint"] = timepoint
        df_plot = df_plot.sort_values(by="mean_abs_value", ascending=False).head(5)
        shap_list.append(df_plot)

In [ ]:
df_plot = pd.concat(shap_list)
df_plot['mean_abs_value_str'] = "+" + df_plot['mean_abs_value'].map('{:.3f}'.format)
clinical_endpoints_description = [val["description"] for val in clinical_endpoints.values()]
df_plot["endpoint"] = pd.Categorical(df_plot["endpoint"], categories=clinical_endpoints_description, ordered=True)
df_plot["timepoint"] = df_plot["timepoint"].div(12).astype(int)

df_plot = df_plot.sort_values(by=["endpoint", "timepoint", "mean_abs_value"], ascending=True)
df_plot["interaction_unique"] = df_plot["interaction"] + "_" + df_plot["endpoint"].astype(str) + "_" + df_plot["timepoint"].astype(str)
df_plot["interaction_unique"] = pd.Categorical(df_plot["interaction_unique"], categories=df_plot["interaction_unique"].unique(), ordered=True)

x = "interaction_unique"
y = "mean_abs_value"
fill_by = "order"
facet_cols = ["endpoint", "timepoint"]

p_shap = (
    p9.ggplot(df_plot, p9.aes(x=x, y=y, fill=fill_by))
    + p9.geom_col()
    + p9.geom_text(p9.aes(label="mean_abs_value_str", color=fill_by), va="center", ha="left", show_legend=False)
    + p9.coord_flip()
    + p9.facet_wrap(facet_cols, scales="free", nrow=1, labeller=p9.labeller(endpoint=lambda x: str(x), timepoint=lambda x: f"{x} years", multi_line=False))
    + p9.theme_classic()
    + p9.theme(strip_background=p9.element_blank(), legend_position="top", axis_title_y=p9.element_blank(), legend_box_spacing=0.01)
    + p9.scale_fill_brewer(type="qualitative", palette="Pastel1", direction=1)
    + p9.scale_color_brewer(type="qualitative", palette="Set1", direction=1)
    + p9.scale_x_discrete(labels=lambda labels: [l.split("_")[0].replace(" ", "\n").replace("PV\n", "PV ") for l in labels])
    + p9.scale_y_continuous(expand=(0, 0, 0.4, 0.)) 
    + p9.labs(y="Shapley Values", tag="d", title="Global feature importance and interactions", fill="Interaction order")
)
p_shap + p9.theme(figure_size=(18, 6))

In [ ]:
pid_results = os.path.join("results_v2_permodality", "clustering", "pid_results.csv")
pid_results = pd.read_csv(pid_results)
pid_results["endpoint"] = pid_results["endpoint"].apply(lambda x: clinical_endpoints[x]["description"])
pid_results["endpoint"] = pd.Categorical(pid_results["endpoint"], categories=endpoints, ordered=True)
print(f"{pid_results.shape}")
pid_results.head()

In [ ]:
x = "timepoint"
y = "Value"
fill_by = "Quantity"
facet_cols = ["endpoint"]

to_plot = pid_results[pid_results["Information"].gt(0) & pid_results["n_clusters"].eq(25)]
to_plot[["Redundancy", "Uniqueness1", "Uniqueness2", "Synergy"]] = to_plot[["Redundancy", "Uniqueness1", "Uniqueness2", "Synergy"]].mul(to_plot["Information"], axis=0)
to_plot = to_plot.groupby(["endpoint", "timepoint", "n_clusters"], as_index=False).mean().drop(columns="fold")
to_plot = pd.melt(to_plot, id_vars=["endpoint", x, "n_clusters"],
        value_vars=["Information", "Redundancy", "Uniqueness1", "Uniqueness2", "Synergy"],
        var_name=fill_by, value_name=y,
       )
to_plot["Quantity"] = to_plot["Quantity"].replace({
    "Uniqueness1": f"Uniqueness ({text_types[0]})",
    "Uniqueness2": f"Uniqueness ({text_types[1]})",
})

p_pid = (
    p9.ggplot(to_plot, p9.aes(x=x, y=y, fill=fill_by, color=fill_by))
    # + p9.geom_line()
    + p9.geom_area(alpha=0.3)
    + p9.facet_wrap(facet_cols, nrow=1)
    + p9.scale_color_brewer(type="qualitative", palette="Dark2", direction=1)
    + p9.scale_fill_brewer(type="qualitative", palette="Dark2", direction=1)
    + p9.theme_classic()
    + p9.theme(strip_background=p9.element_blank(), legend_position="top", legend_box_spacing=0.01, axis_title_x=p9.element_text(margin={'t': -.3, 'units': 'pt'}))
    + p9.labs(tag="c", x="Time (months)", y="Estimation", title="Multimodal data statistics over time")
)
p_pid

In [ ]:
source_data = pid_results[pid_results["Information"].gt(0) & pid_results["n_clusters"].eq(25)].drop(columns="n_clusters")
source_data[["Redundancy", "Uniqueness1", "Uniqueness2", "Synergy"]] = source_data[["Redundancy", "Uniqueness1", "Uniqueness2", "Synergy"]].mul(source_data["Information"], axis=0)
with pd.ExcelWriter(os.path.join("source_data", "Source Data S5.xlsx"), engine='openpyxl', mode="a") as writer:
    source_data.to_excel(writer, sheet_name='Raw Results (Figure 6c)', index=False)
    source_data.groupby(["endpoint", "timepoint"], as_index=False). \
        apply(lambda x: x.mean().round(2).astype(str).add("\u00B1").add(x.mean().round(2).astype(str))). \
        drop(columns="fold"). \
        to_excel(writer, sheet_name= "Figure6c", index=False)

In [ ]:
composition = p_coeff / (p_comp / p_pid) / p_shap + p9.theme(figure_size=(12, 14))
composition

In [ ]:
composition.save(os.path.join("figures", "Figure6.svg"))
composition.save(os.path.join("figures", "Figure6.pdf"))